# Trabalho Prático: Aprendizado de Máquina (Supervisionado e Não Supervisionado)
**Dataset:** QSAR Biodegradation (1054 amostras, 41 features moleculares)
**Objetivo:** Classificação binária de substâncias químicas (Prontamente Biodegradável - RB vs Não Prontamente Biodegradável - NRB).

## Estrutura do Notebook:
1. Configuração e Importação de Bibliotecas
2. Leitura, Limpeza e Normalização Obrigatória dos Dados
3. Divisão dos Dados (70% Treinamento / 15% Validação / 15% Teste)
4. Modelagem Supervisionada (Decision Tree, KNN, Artificial Neural Network) com otimização
5. Modelagem Não Supervisionada (K-Means) com Método do Cotovelo e Silhouette Score
6. Análise de Componentes Principais (PCA) para visualização dos Clusters

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Divisão
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos Supervisionados
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Clusterização
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Métricas de Avaliação
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, silhouette_score)

from imblearn.over_sampling import SMOTE

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
import warnings
warnings.filterwarnings('ignore') # Ignorar avisos de depreciação menores

## 1. Carregamento, Limpeza e Normalização dos Dados
Mesmo que o arquivo original já contenha a palavra "normalizado", passaremos os dados por um pipeline rigoroso de verificação de nulos, separação de variáveis e padronização estatística (z-score). O resultado será exportado para um novo arquivo CSV limpo, cumprindo o requisito de ter um script de pré-processamento.

In [ ]:
# 1.1 Carregamento dos dados
# Lendo o arquivo original (certifique-se de que o arquivo está no mesmo diretório do notebook)
# Assumindo separador por vírgula com base nos metadados da amostra
df_raw = pd.read_csv('biodeg_normalizado.csv')

print(f"Dimensões originais do dataset: {df_raw.shape}")

# 1.2 Limpeza de Dados (Tratamento de Nulos/Duplicatas)
# Remoção de duplicatas se existirem
df_raw = df_raw.drop_duplicates()
# Remoção de valores nulos (se existirem)
df_raw = df_raw.dropna()
print(f"Dimensões após limpeza básica: {df_raw.shape}")

# 1.3 Separação de Features (X) e Target (y)
target_col = 'experimental class' # Conforme inspecionado no cabeçalho dos dados
X_raw = df_raw.drop(columns=[target_col])
y_raw = df_raw[target_col]

# 1.4 Codificação da Variável Alvo
# RB (Ready Biodegradable) e NRB (Not Ready Biodegradable)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
# Mapeamento para conferência: classes originais para valores numéricos
le_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Mapeamento do Target: {le_mapping}")

# 1.5 Normalização Estatística (Z-Score)
# É crucial para o KNN, K-Means e Redes Neurais que as features estejam na mesma escala
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# 1.6 Salvando o Dataset Final Processado
df_clean = X_scaled.copy()
df_clean['target_encoded'] = y_encoded
df_clean.to_csv('biodeg_limpo_e_processado.csv', index=False)
print("\nDataset limpo e normalizado salvo com sucesso como 'biodeg_limpo_e_processado.csv'.")

## 2. Divisão dos Dados (Holdout Estratificado: 70/15/15)
O dataset é desbalanceado (~66% NRB vs ~34% RB). Portanto, a estratificação é vital para garantir que a mesma proporção de classes exista no treino, validação e teste.

In [ ]:
# Para obter 70% Treino, 15% Validação e 15% Teste:
# Primeiro passo: Extraímos 30% do total para um conjunto temporário (que será dividido ao meio)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42
)

# Segundo passo: Dividimos os 30% temporários em 50/50, resultando em 15% de validação e 15% de teste reais
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Tamanho do Conjunto de Treino:    {X_train.shape[0]} amostras ({X_train.shape[0]/df_clean.shape[0]*100:.1f}%)")
print(f"Tamanho do Conjunto de Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/df_clean.shape[0]*100:.1f}%)")
print(f"Tamanho do Conjunto de Teste:     {X_test.shape[0]} amostras ({X_test.shape[0]/df_clean.shape[0]*100:.1f}%)")

# =====================================================================
# APLICAÇÃO DO SMOTE (Apenas no conjunto de TREINO)
# =====================================================================
print("\n--- Aplicando SMOTE no conjunto de Treino ---")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Verificação do novo balanceamento
unique_smote, counts_smote = np.unique(y_train_smote, return_counts=True)
print(f"Novo tamanho do Treino (após SMOTE): {X_train_smote.shape[0]} amostras")
print(f"Nova proporção das classes no Treino: {dict(zip(unique_smote, counts_smote/len(y_train_smote)))}")
# =====================================================================

# Verificação do balanceamento no teste
unique, counts = np.unique(y_test, return_counts=True)
print(f"\nProporção das classes no Teste: {dict(zip(unique, counts/len(y_test)))}")

## 3. Modelagem Supervisionada (Classificação)
Testaremos 3 algoritmos:
1. **Decision Tree (Árvore de Decisão)**
2. **KNN (K-Nearest Neighbors)**
3. **Artificial Neural Network (Multi-Layer Perceptron)**

Para garantir o uso *exato* do conjunto de Validação para a escolha dos melhores hiperparâmetros (e não usar cross-validation interna no treino), utilizaremos o `PredefinedSplit`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# =====================================================================
# 1. SELEÇÃO DE FEATURES (Reduzindo de 41 para 20)
# =====================================================================
print("--- Análise de Importância das Features (Random Forest) ---")

# Treinamos uma Random Forest apenas nos dados de treino balanceados (SMOTE)
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_train_smote, y_train_smote)

# Extraímos a importância matemática e criamos um DataFrame
importancias = rf_selector.feature_importances_
df_importancias = pd.DataFrame({
    'Feature': X_train.columns,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=False)

print(df_importancias)

# Plotamos as 20 Features mais importantes (Use este gráfico no seu relatório!)
plt.figure(figsize=(10, 8))
sns.barplot(x='Importancia', y='Feature', data=df_importancias.head(20), palette='viridis')
plt.title('Top 20 Features Mais Importantes para Biodegradabilidade')
plt.xlabel('Grau de Importância')
plt.ylabel('Descritor Molecular')
plt.tight_layout()
plt.show()

# Selecionamos apenas os nomes das Top 20 features
top_20_features = df_importancias['Feature'].head(20).tolist()
print(f"\nReduzindo os dados de 41 para as 20 features mais importantes...")

# Sobrescrevemos os conjuntos mantendo SÓ as 20 colunas
X_train_smote_red = X_train_smote[top_20_features]
X_val_red = X_val[top_20_features]
X_test_red = X_test[top_20_features]

# =====================================================================
# 2. CONSTRUINDO O PREDEFINED SPLIT (Com os dados reduzidos)
# =====================================================================
# Concatenamos Treino e Validação (AGORA COM AS 20 COLUNAS)
X_train_val = pd.concat([X_train_smote_red, X_val_red], axis=0)
y_train_val = np.concatenate([y_train_smote, y_val]) # O target (y) não muda

# -1 indica TREINO, 0 indica VALIDAÇÃO
test_fold = np.zeros(X_train_val.shape[0])
test_fold[:X_train_smote_red.shape[0]] = -1
ps = PredefinedSplit(test_fold)

# =====================================================================
# 3. FUNÇÃO DE AVALIAÇÃO ATUALIZADA
# =====================================================================
resultados_modelos = {}

def avaliar_modelo(nome, modelo, params):
    """
    Função utilitária atualizada para usar os dados reduzidos (20 features).
    """
    print(f"--- Treinando e Otimizando: {nome} ---")
    grid = GridSearchCV(modelo, params, cv=ps, scoring='f1_macro', n_jobs=-1)

    # Treina com os dados reduzidos
    grid.fit(X_train_val, y_train_val)

    best_model = grid.best_estimator_
    print(f"Melhores Hiperparâmetros: {grid.best_params_}")

    # Previsão no conjunto de TESTE REDUZIDO
    y_pred = best_model.predict(X_test_red)

    # Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    resultados_modelos[nome] = {'Acurácia': acc, 'F1-Score (Macro)': f1}

    print("\nRelatório de Classificação no Conjunto de TESTE (20 Features):")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    # Plotagem da Matriz de Confusão
    plt.figure(figsize=(5,4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Matriz de Confusão - {nome} (20 Features)')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.show()
    print("="*60 + "\n")
    return best_model

## 4. Modelagem Não Supervisionada (K-Means)
Vamos esquecer temporariamente os rótulos originais (`y`) e verificar como os dados moleculares se agrupam naturalmente. Usaremos toda a base padronizada (`X_scaled`).

Conforme especificado, iteraremos com $K$ variando de 2 a 10 e utilizaremos duas abordagens em conjunto para decidir o melhor $K$:
1. **Método do Cotovelo (Inércia / WCSS):** Busca o ponto onde a adição de novos clusters deixa de reduzir significativamente a variância interna.
2. **Silhouette Score:** Mede o quão similar um objeto é ao seu próprio cluster em comparação com outros clusters (varia de -1 a 1, sendo quanto mais próximo de 1, melhor).

In [ ]:
inercia = []
silhouette_scores = []
K_range = range(2, 11) # K entre 2 e 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)

    inercia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plotando os Gráficos Lado a Lado
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Cotovelo (Inércia)
ax[0].plot(K_range, inercia, marker='o', color='blue', linestyle='--')
ax[0].set_title('Método do Cotovelo (Inércia)')
ax[0].set_xlabel('Número de Clusters (k)')
ax[0].set_ylabel('Inércia (WCSS)')
ax[0].set_xticks(K_range)

# Gráfico 2: Silhouette Score
ax[1].plot(K_range, silhouette_scores, marker='s', color='green', linestyle='-')
ax[1].set_title('Silhouette Score')
ax[1].set_xlabel('Número de Clusters (k)')
ax[1].set_ylabel('Score de Silhueta')
ax[1].set_xticks(K_range)

plt.tight_layout()
plt.show()

### Análise e Execução do Melhor K
Em problemas binários, frequentemente testamos `K=2` para ver se os clusters refletem as classes originais (RB e NRB). No entanto, deves inspecionar visualmente o Silhouette e o Cotovelo. Para fins didáticos e baseando-se no comportamento comum deste dataset, aplicaremos `K=2` e compararemos os clusters formados de forma não-supervisionada com os rótulos reais.

In [ ]:
# Supondo k ótimo (exemplo: k=2 baseado na semântica do problema ou nos gráficos)
melhor_k = 2
print(f"Treinando K-Means com k={melhor_k}...")

kmeans_final = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
clusters_pred = kmeans_final.fit_predict(X_scaled)

# Comparando os clusters descobertos com a classe experimental real
df_comparacao = pd.DataFrame({
    'Classe_Real': label_encoder.inverse_transform(y_encoded),
    'Cluster_Atribuido': clusters_pred
})

crosstab = pd.crosstab(df_comparacao['Classe_Real'], df_comparacao['Cluster_Atribuido'],
                       rownames=['Classe Original'], colnames=['Cluster Encontrado'])
print("\nTabela de Contingência (Original vs Cluster):")
display(crosstab)

print("\nNota: Como o aprendizado é não-supervisionado, os rótulos do cluster (0 e 1) "
      "podem estar invertidos em relação aos rótulos originais. O que importa é a pureza "
      "de cada cluster (se ele isolou bem a maioria das instâncias RB de um lado e NRB do outro).")

## 5. Visualização dos Clusters (Redução de Dimensionalidade com PCA)
O dataset possui 41 dimensões, o que impede a visualização direta. Usaremos a Análise de Componentes Principais (PCA) para comprimir as 41 features em 2 componentes e enxergar a distribuição espacial dos clusters formados pelo K-Means.

In [ ]:
# Aplicando PCA para reduzir para 2 dimensões
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Criando dataframe para plotagem
df_pca = pd.DataFrame(data = X_pca, columns = ['Componente 1', 'Componente 2'])
df_pca['Cluster'] = clusters_pred
df_pca['Classe Real'] = label_encoder.inverse_transform(y_encoded)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Clusters do K-Means
sns.scatterplot(x='Componente 1', y='Componente 2', hue='Cluster',
                palette='Set1', data=df_pca, ax=ax[0], alpha=0.7)
ax[0].set_title('Clusters encontrados pelo K-Means')

# Plot 2: Classes Reais
sns.scatterplot(x='Componente 1', y='Componente 2', hue='Classe Real',
                palette='Set2', data=df_pca, ax=ax[1], alpha=0.7)
ax[1].set_title('Distribuição Real das Classes (RB vs NRB)')

plt.tight_layout()
plt.show()

print(f"Variância explicada pelos 2 componentes principais: {pca.explained_variance_ratio_.sum()*100:.2f}%")